## 1. Ajeitar o método de FDI
## 2. Trains crescentes e partindo do zero

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Lista mestre para guardar os DataFrames de todas as portas já processadas
todas_as_portas = [] 

nomes_colunas = [
    "Duration", "Ref_Pos", "Fbk_Pos", "Ref_Vel", "Fbk_Vel", 
    "Fbk_Hall", "Fbk_Enc", "Vbus", "Temp_Motor", 
    "Corrente_A", "Corrente_B", "Corrente_C", 
    "Temp_Driver", "Volt_A", "Volt_B", "Volt_C"
]

# Loop para N portas
for i in range(1, 49):  # Range arbitrário, pode ser qualquer quantidade
    print(f"Analisando o Train_{i}...")  
    caminho_pasta = f"Train/Train_{i}/"
    arquivo_rul = f"{caminho_pasta}F_{i}_RUL.csv"
    
    try:
        with open(arquivo_rul, 'r') as f:
            linha = f.readline()
            nCiclos = int(linha.strip())
    except FileNotFoundError:
        print(f"Erro: Arquivo RUL não encontrado na porta {i}")
        continue # Pula para a próxima porta se der erro

    dados_processados = []
    
    # Loop pelos ciclos da porta atual
    for j in range(1, nCiclos + 1):
        file_open = f"{caminho_pasta}F_{i}_{j:05d}_Opening.csv"
        
        if not os.path.exists(file_open):
            continue
            
        df_open = pd.read_csv(file_open, sep=';', header=None, names=nomes_colunas)
        
        x_max_atual = df_open['Fbk_Pos'].iloc[0] 
        
        dados_processados.append({
            'Ciclo': j,
            'X_max_Atual': x_max_atual
        })
    
    # Se não processou nenhum dado (arquivos faltando), pula
    if not dados_processados:
        continue
        
    # Transforma os dados da porta atual em DataFrame
    df_porta = pd.DataFrame(dados_processados)
    df_porta = df_porta.sort_values(by='Ciclo').reset_index(drop=True)
    
    # ----------------------------------------------
    # NOVO MÉTODO DE ACHAR O FDI
    # ----------------------------------------------
    
    limite_head = min(15, len(df_porta))
    referencia = df_porta['X_max_Atual'].head(limite_head).median() # Calcula a referência (mediana dos 15 primeiros ciclos)
    print(f"Referencia: {referencia}")
    limiar_queda = referencia - 7.1 # Define o que é considerado uma "queda" (menos que 7 não detectaria corretamente no Train_4; vou melhorar no futuro pra n ser um valor fixo)

    fdi_ciclo = None
    
    # 2. Busca pelo verdadeiro FDI ciclo a ciclo
    for idx, row in df_porta.iterrows():
        if row['X_max_Atual'] < limiar_queda:
            
            # Extrai todos os ciclos que acontecem DEPOIS deste (o "futuro")
            ciclos_futuros = df_porta.loc[idx+1:, 'X_max_Atual']
            
            # Cria uma série dizendo True (voltou) ou False (não voltou) para cada ciclo futuro
            recuperacoes = (ciclos_futuros >= referencia)
            
            # Desliza uma janela somando os valores (True vale 1, False vale 0).
            # Se a soma chegar a 3, tivemos 3 ciclos saudáveis consecutivos.
            teve_recuperacao_sustentada = (recuperacoes.rolling(window=3).sum() == 3).any() # Ou seja, considera caso X_max aumente até a referência e permaneça por pelo menos 3 ciclos
            
            # Se a porta provar que voltou ao normal de forma sustentada...
            if teve_recuperacao_sustentada:
                # ...o tombo inicial foi só um ruído/falso positivo. Ignora e continua a busca!
                continue 
            else:
                # Se ela caiu e NÃO conseguiu emendar 3 ciclos normais no futuro, é o FDI real!
                # -1 pois estamos considerando o FDI como o último ciclo antes da queda, e não o primeiro depois da queda
                fdi_ciclo = row['Ciclo'] - 1
                break
    
    # Cria a coluna relativa baseada nesse FDI verificado
    if fdi_ciclo is not None:
        print(f"Train {i} | Último ciclo saudável: {fdi_ciclo}")
        df_porta['Ciclo_Relativo'] = df_porta['Ciclo'] - fdi_ciclo
        
        # Calcula a distância: se a ref é 190 e o atual é 189, a distância (degradação) é 1.0
        df_porta['Distancia_Ref'] = referencia - df_porta['X_max_Atual']
        
        df_porta_filtrada = df_porta[(df_porta['Ciclo_Relativo'] >= -5)]
        todas_as_portas.append((i, df_porta_filtrada))

Analisando o Train_1...
Referencia: 190.0
Train 1 | Último ciclo saudável: 1593.0
Analisando o Train_2...
Referencia: 189.0
Train 2 | Último ciclo saudável: 1135.0
Analisando o Train_3...
Referencia: 189.0
Train 3 | Último ciclo saudável: 1316.0
Analisando o Train_4...
Referencia: 189.0
Train 4 | Último ciclo saudável: 2785.0
Analisando o Train_5...
Referencia: 190.0
Train 5 | Último ciclo saudável: 736.0
Analisando o Train_6...
Referencia: 190.0
Train 6 | Último ciclo saudável: 219.0
Analisando o Train_7...
Referencia: 190.0
Train 7 | Último ciclo saudável: 373.0
Analisando o Train_8...
Referencia: 190.0
Train 8 | Último ciclo saudável: 629.0
Analisando o Train_9...
Referencia: 189.0
Train 9 | Último ciclo saudável: 1526.0
Analisando o Train_10...
Referencia: 189.0
Train 10 | Último ciclo saudável: 1458.0
Analisando o Train_11...
Referencia: 188.0
Train 11 | Último ciclo saudável: 1126.0
Analisando o Train_12...
Referencia: 188.0
Train 12 | Último ciclo saudável: 866.0
Analisando o Tr

### Gráfico

In [3]:
import plotly.graph_objects as go
import plotly.colors as pc
import numpy as np

# Inicializa a figura em branco
fig = go.Figure()

num_linhas = len(todas_as_portas)

# Cria a paleta de cores 'turbo' fatiada matematicamente para o Plotly
valores_cor = np.linspace(0, 1, num_linhas)
paleta_cores = pc.sample_colorscale('turbo', valores_cor)

# Adiciona cada trem como uma linha (trace) interativa
for idx, (porta_id, df_plot) in enumerate(todas_as_portas):
    fig.add_trace(go.Scatter(
        x=df_plot['Ciclo_Relativo'],
        y=df_plot['Distancia_Ref'],
        mode='lines',
        line=dict(color=paleta_cores[idx], width=1.5),
        opacity=0.6, # Transparência equivalente ao alpha
        name=f'Train {porta_id}',
        # Configura a caixinha interativa que aparece ao passar o mouse:
        hovertemplate=f'<b>Train {porta_id}</b><br>Ciclo FDI: %{{x}}<br>Perda: %{{y:.2f}}°<extra></extra>'
    ))

# ---------------------------------------------------------
# Linhas de Marcação (Limiar e Ciclo Zero)
# ---------------------------------------------------------
# O Plotly tem funções específicas para cruzar o gráfico com linhas retas:
fig.add_hline(y=0, line_dash="dash", line_color="red", line_width=1.5)
fig.add_vline(x=0, line_dash="solid", line_color="black", line_width=1, opacity=0.7)

# Para que essas linhas apareçam na legenda (como no Matplotlib), 
# criamos "linhas invisíveis" apenas com o nome para ancorar na legenda
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='red', width=2, dash='dash'), name='Limiar de Falha'))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='black', width=1.5), name='Último Ciclo Saudável'))

# ---------------------------------------------------------
# Embelezamento e Layout (equivalente ao plt.title, labels e grid)
# ---------------------------------------------------------
fig.update_layout(
    title=dict(text='Curvas de Degradação Normalizadas', font=dict(size=20)),
    xaxis_title='Ciclos FDI',
    yaxis_title='Perda de Amplitude da Porta',
    
    # Tamanho da figura (equivalente ao figsize=(16,7))
    width=1200, 
    height=600,
    
    # Fundo branco com grid sutil (igual ao matplotlib com seaborn)
    template="plotly_white",
    hovermode="closest", # O mouse foca no ponto mais próximo
    
    # Configuração da Legenda (coloca do lado de fora, com scrollbar automático se ficar grande)
    legend=dict(
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02,
        font=dict(size=10)
    )
)

# Reforça as linhas de grade pontilhadas
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', griddash='dot')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', griddash='dot')

# Exibe o gráfico interativo
fig.show()